# Generating Synthetic Genomes with N-gram (Markov Chain) Models
**Approach:** k-mer based Markov chain modeling of genomic sequence  
**Validation:** GC content comparison, k-mer distribution analysis, Jensen-Shannon divergence

---

## Overview

N-gram models are a foundational technique in natural language processing, where the next word in a sequence is predicted based on the previous N-1 words. The same idea applies directly to genomic sequences: instead of words, we use **k-mers** (short DNA subsequences of length k), and instead of predicting the next word, we predict the next nucleotide.

This project builds a k-mer-based Markov chain model from a real bacterial genome, uses it to generate entirely synthetic DNA sequences, and then statistically validates how closely those synthetic sequences resemble the real genome's underlying sequence composition.

**Pipeline:**
1. Parse a real genome assembly (FASTA format)
2. Build a k-mer → next-nucleotide transition model (the "N-gram" model)
3. Convert transition counts into probabilities
4. Generate synthetic genome sequences by sampling from this model
5. Validate the synthetic sequences against the real genome using GC content and k-mer distribution comparisons
6. Quantify similarity using Jensen-Shannon divergence

**Why this matters biologically:** genomic sequences are not random — local sequence composition (GC content, codon bias, dinucleotide frequencies) carries real biological signal shaped by evolutionary pressure. A model that can capture and reproduce these local statistical patterns demonstrates a genuine, quantifiable understanding of what makes a genome "look like" a genome, rather than random noise.

## 1. Setup and Data Loading

We use a bacterial genome assembly (NCBI accession `NZ_ACYS00000000`) as our real-world training sequence. The file is provided as a `.tar` archive containing a FASTA-format scaffold file.

In [ ]:
import tarfile
import os
import random
import math
from collections import defaultdict, Counter

# Inspect the archive before extracting
!tar -tf NZ_ACYS00000000.scaffold.fna.tar

In [ ]:
# Extract the genome archive
tar_path = "NZ_ACYS00000000.scaffold.fna.tar"

with tarfile.open(tar_path, "r") as tar:
    tar.extractall("genome_data")

print("Extracted files:", os.listdir("genome_data"))

### 1.1 Parse the FASTA file

FASTA files can contain multiple sequences (scaffolds/contigs), each starting with a `>` header line. We parse all sequences into a list, concatenating multi-line sequence blocks as needed.

In [ ]:
def read_fasta(filepath):
    """
    Parse a FASTA file into a list of sequence strings.

    Parameters
    ----------
    filepath : str — path to a FASTA file

    Returns
    -------
    list of str — one entry per sequence/scaffold in the file
    """
    sequences = []
    seq = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    sequences.append("".join(seq))
                    seq = []
            else:
                seq.append(line.upper())
        if seq:
            sequences.append("".join(seq))
    return sequences

In [ ]:
# Locate the extracted FASTA file (extension may be .fna or .fa)
fasta_path = None
for root, dirs, files in os.walk("genome_data"):
    for file in files:
        if file.endswith(".fna") or file.endswith(".fa"):
            fasta_path = os.path.join(root, file)

genomes = read_fasta(fasta_path)

print(f"Number of scaffolds: {len(genomes)}")
print(f"First scaffold length: {len(genomes[0]):,} bp")

## 2. Preprocessing

Raw genome assemblies sometimes contain ambiguous bases (e.g., `N` for unknown nucleotides) or lowercase soft-masked regions. We clean each sequence down to the four standard bases (A, C, G, T) only, since our model only knows how to handle these.

In [ ]:
def clean_sequence(seq):
    """Keep only standard A/C/G/T bases, discarding ambiguous codes (N, etc.)."""
    return "".join(b for b in seq if b in "ACGT")

genomes = [clean_sequence(seq) for seq in genomes]
print(f"Cleaned scaffold lengths: {[len(g) for g in genomes[:5]]} ...")

## 3. Building the K-mer Transition Model

The core idea: for every position in the genome, look at the preceding **k** nucleotides (the "k-mer" or context) and record which nucleotide comes next. Across the whole genome, this builds up a table of how often each possible next-base follows each possible k-mer context — exactly analogous to an N-gram language model, where k-mers play the role of "words."

With k=3, there are 4³ = 64 possible 3-mer contexts, each potentially followed by any of 4 bases.

In [ ]:
def build_ngram_model(sequences, k=3):
    """
    Build a k-mer -> next-base transition count model.

    Parameters
    ----------
    sequences : list of str — cleaned genome sequences
    k : int — context length (k-mer size)

    Returns
    -------
    dict of dict — model[kmer][next_base] = count
    """
    model = defaultdict(lambda: defaultdict(int))

    for seq in sequences:
        if len(seq) <= k:
            continue
        for i in range(len(seq) - k):
            kmer = seq[i:i + k]
            next_base = seq[i + k]
            model[kmer][next_base] += 1

    return model

In [ ]:
k = 3
ngram_model = build_ngram_model(genomes, k)

print(f"Unique {k}-mer contexts observed: {len(ngram_model)}")
print(f"(Maximum possible: 4^{k} = {4**k})")

## 4. Converting Counts to Transition Probabilities

Raw counts tell us *how often* a transition occurred, but to generate new sequences we need *probabilities* — for each k-mer context, what fraction of the time is each possible next base observed in the real genome?

In [ ]:
def normalize_model(model):
    """Convert raw transition counts into probabilities that sum to 1 per context."""
    prob_model = {}
    for kmer, next_bases in model.items():
        total = sum(next_bases.values())
        prob_model[kmer] = {
            base: count / total
            for base, count in next_bases.items()
        }
    return prob_model

prob_model = normalize_model(ngram_model)

# Sanity check — probabilities for one example k-mer context
example_kmer = list(prob_model.keys())[0]
print(f"Example context '{example_kmer}':")
print(prob_model[example_kmer])
print(f"Sum of probabilities: {sum(prob_model[example_kmer].values()):.4f}  (should be 1.0)")

## 5. Generating Synthetic Genomes

With a trained probability model in hand, we can generate entirely new sequences: starting from a random seed k-mer, repeatedly sample the next base according to the learned probabilities for the current trailing k-mer context, then slide the window forward and repeat.

In [ ]:
def sample_next_base(prob_dict):
    """Randomly sample a base according to a probability distribution dict."""
    bases = list(prob_dict.keys())
    probs = list(prob_dict.values())
    return random.choices(bases, probs)[0]


def generate_genome(prob_model, k=3, length=50_000):
    """
    Generate a synthetic genome sequence by sampling from a trained k-mer model.

    Parameters
    ----------
    prob_model : dict — trained kmer -> {base: probability} model
    k : int — context length used to train the model
    length : int — desired length of the generated sequence

    Returns
    -------
    str — synthetic genome sequence
    """
    seed = random.choice(list(prob_model.keys()))
    genome = seed

    for _ in range(length - k):
        context = genome[-k:]
        if context not in prob_model:
            # Fallback: context never seen in training data, pick a random base
            genome += random.choice("ACGT")
        else:
            genome += sample_next_base(prob_model[context])

    return genome

In [ ]:
# Generate 5 independent synthetic genomes
synthetic_genomes = [generate_genome(prob_model, k, length=50_000) for _ in range(5)]

for i, genome in enumerate(synthetic_genomes):
    print(f"Synthetic genome {i+1}")
    print(f"  Length:      {len(genome):,} bp")
    print(f"  First 60bp:  {genome[:60]}")

# Confirm all 5 are genuinely distinct sequences
print(f"\nUnique genomes generated: {len(set(synthetic_genomes))} / {len(synthetic_genomes)}")

### 5.1 Export to FASTA format

The synthetic genomes are saved in standard FASTA format, making them usable with any downstream bioinformatics tool that expects this format.

In [ ]:
with open("synthetic_genomes.fasta", "w") as f:
    for i, genome in enumerate(synthetic_genomes):
        f.write(f">synthetic_genome_{i+1}\n")
        for j in range(0, len(genome), 60):
            f.write(genome[j:j+60] + "\n")

print("Saved synthetic genomes to synthetic_genomes.fasta")

# Preview FASTA-formatted output
print("\nFASTA preview (genome 1, first 3 lines):")
for i, genome in enumerate(synthetic_genomes[:1]):
    print(f">synthetic_genome_{i+1}")
    for j in range(0, 180, 60):
        print(genome[j:j+60])

## 6. Statistical Validation — How Realistic Are the Synthetic Genomes?

Generating sequences is easy; the real question is whether they actually capture meaningful biological structure from the source genome. We validate this in three ways: GC content, k-mer distribution similarity, and a formal divergence metric.

### 6.1 GC Content Comparison

GC content (the proportion of G and C bases) is one of the most basic descriptive statistics of a genome, and varies meaningfully between species and even between regions of the same genome. If our model preserves real biological signal, synthetic genomes should have GC content close to the real source genome.

In [ ]:
def gc_content(seq):
    """Compute the fraction of G/C bases in a sequence."""
    gc = sum(1 for base in seq if base in "GC")
    return gc / len(seq)

real_gc = gc_content(genomes[0])
synthetic_gc = [gc_content(seq) for seq in synthetic_genomes]

print(f"Real genome GC content:       {real_gc:.4f}")
print(f"Synthetic genome GC content:  {[round(g, 4) for g in synthetic_gc]}")
print(f"Mean synthetic GC content:    {sum(synthetic_gc)/len(synthetic_gc):.4f}")
print(f"Difference from real:         {abs(real_gc - sum(synthetic_gc)/len(synthetic_gc)):.4f}")

### 6.2 K-mer Distribution Comparison

GC content alone is a coarse summary — two very different sequences could share the same GC content by coincidence. A more sensitive test compares the full distribution of k-mer frequencies between real and synthetic sequences.

In [ ]:
def kmer_distribution(seq, k=3):
    """Count occurrences of every k-mer in a sequence."""
    return Counter(seq[i:i+k] for i in range(len(seq) - k))


def normalize_counter(counter):
    """Convert raw k-mer counts into a probability distribution."""
    total = sum(counter.values())
    return {kmer: count / total for kmer, count in counter.items()}


real_kmers = kmer_distribution(genomes[0], k)
synthetic_kmers = kmer_distribution(synthetic_genomes[0], k)

real_kmers_norm = normalize_counter(real_kmers)
synthetic_kmers_norm = normalize_counter(synthetic_kmers)

print(f"Unique {k}-mers in real genome:      {len(real_kmers_norm)}")
print(f"Unique {k}-mers in synthetic genome: {len(synthetic_kmers_norm)}")

In [ ]:
import matplotlib.pyplot as plt

# Compare the top 15 most common k-mers in the real genome against
# the synthetic genome's frequency for those same k-mers
top_real_kmers = sorted(real_kmers_norm, key=real_kmers_norm.get, reverse=True)[:15]

real_freqs = [real_kmers_norm[kmer] for kmer in top_real_kmers]
synth_freqs = [synthetic_kmers_norm.get(kmer, 0) for kmer in top_real_kmers]

x = range(len(top_real_kmers))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar([i - width/2 for i in x], real_freqs, width, label='Real genome', color='steelblue')
plt.bar([i + width/2 for i in x], synth_freqs, width, label='Synthetic genome', color='darkorange')
plt.xticks(x, top_real_kmers, rotation=45)
plt.ylabel('Frequency')
plt.title(f'Top 15 Most Common {k}-mers — Real vs. Synthetic Genome')
plt.legend()
plt.tight_layout()
plt.show()

### 6.3 Jensen-Shannon Divergence

To quantify similarity between the two full k-mer distributions with a single number, we use **Jensen-Shannon divergence (JSD)** — a symmetric, bounded measure of how different two probability distributions are. JSD ranges from 0 (identical distributions) to 1 (completely disjoint distributions), making it easy to interpret regardless of distribution shape.

In [ ]:
def js_divergence(p, q):
    """
    Compute the Jensen-Shannon divergence between two probability distributions,
    represented as dicts mapping category -> probability.

    Returns a value between 0 (identical) and 1 (maximally different).
    """
    keys = set(p.keys()).union(q.keys())
    p = {k: p.get(k, 0) for k in keys}
    q = {k: q.get(k, 0) for k in keys}
    m = {k: (p[k] + q[k]) / 2 for k in keys}

    def kl_divergence(a, b):
        return sum(a[k] * math.log2(a[k] / b[k]) for k in keys if a[k] > 0 and b[k] > 0)

    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)


js_score = js_divergence(real_kmers_norm, synthetic_kmers_norm)
print(f"Jensen-Shannon divergence (real vs. synthetic {k}-mer distributions): {js_score:.4f}")
print("(0 = identical distributions, 1 = completely different)")

**Interpreting this result:** a low JSD score (close to 0) indicates the synthetic genome's local sequence composition closely mirrors the real genome — exactly what we'd expect, since the model was explicitly trained to replicate k-mer transition probabilities from the source sequence. This is a useful sanity check that the generation process is working correctly, though it's worth noting that low JSD on k-mer frequency alone does not imply the synthetic sequence captures longer-range biological structure (genes, regulatory elements, codon usage bias across reading frames, etc.) — only local k-mer-level statistics.

## 7. Summary

This project applied N-gram (Markov chain) modeling — a technique borrowed directly from natural language processing — to genomic sequence data, treating k-mers as the genomic equivalent of "words."

**What we did:**
1. Parsed and cleaned a real bacterial genome assembly
2. Built a k-mer → next-nucleotide transition probability model (a Markov chain of order k)
3. Generated multiple synthetic genome sequences by sampling from this model
4. Validated the synthetic sequences against the real genome using GC content, k-mer frequency comparison, and Jensen-Shannon divergence

**Key takeaways:**
- A surprisingly simple model (just tracking which base follows each k-mer) is sufficient to reproduce local sequence composition statistics like GC content and k-mer frequency distributions
- Jensen-Shannon divergence provides a clean, interpretable, bounded way to quantify distributional similarity — useful well beyond genomics, in any context comparing two probability distributions
- This approach has a direct parallel to language modeling: just as N-gram models predict the next word from previous words, k-mer models predict the next nucleotide from previous nucleotides — the same underlying Markov chain framework applies to both domains

**Limitations and potential extensions:**
- Higher-order models (larger k) would capture longer-range dependencies, at the cost of needing more training data to estimate transition probabilities reliably
- This model captures local statistical structure only — it has no concept of genes, open reading frames, or regulatory elements, so synthetic sequences are not biologically functional
- A natural extension would be comparing model performance (via JSD) across multiple values of k to find where additional context stops meaningfully improving fidelity to the real genome
- Applying the same framework to compare k-mer signatures across different species could be used for taxonomic classification, a real technique used in metagenomics